# Parquet 30-Minute Interval Data Cleanup

This notebook recursively scans through all the parquet files in `/opt/rws/data/fmp/fmp-m30` to check if any rows are missing volume (i.e., volume is `0`). If they are, it drops those rows and re-writes the parquet file. If no rows remain after dropping, the file is deleted.

In [15]:
import os
import glob
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
from tqdm.notebook import tqdm

DATA_DIR = os.path.abspath("/opt/rws/data/fmp/fmp-daily")
print(f"Target Data Directory: {DATA_DIR}")

Target Data Directory: /opt/rws/data/fmp/fmp-daily


## 1. Find all Parquet Files

We locate all `.parquet` files recursively in the target directory.

In [16]:
search_pattern = os.path.join(DATA_DIR, "**", "*.parquet")
parquet_files = glob.glob(search_pattern, recursive=True)
print(f"Found {len(parquet_files):,} parquet files to scan.")

Found 20,984 parquet files to scan.


## 2. Define the Cleanup Function

This function reads a parquet file, identifies and drops rows with zero volume, and rewrites or deletes the file. It also supports `dry_run` mode.

In [17]:
def process_parquet_file(file_path, dry_run=True):
    """
    Processes a single parquet file:
    - Reads file
    - Identifies rows where volume is 0 and drops them
    - Identifies NaNs in OHLC columns and fills them using rules:
        - If low is missing, it is min of other OHLC columns with non-NaN values
        - If high is missing, it is max of other OHLC columns with non-NaN values
        - If open is missing, it takes the value of low
        - If close is missing, it takes the value of high
    - Drops any rows where all OHLC values remain NaN (or are missing)
    - Rewrites the file (or deletes it if empty)
    - Returns a dictionary detailing the action taken
    """
    try:
        df = pd.read_parquet(file_path)
    except Exception as e:
        return {
            "file": file_path,
            "status": "error_reading",
            "error": str(e),
            "rows_dropped": 0,
            "nans_filled": 0
        }
    
    if df.empty:
        if not dry_run:
            try:
                os.remove(file_path)
            except Exception as e:
                return {
                    "file": file_path,
                    "status": "error_deleting",
                    "error": str(e),
                    "rows_dropped": 0,
                    "nans_filled": 0
                }
        return {
            "file": file_path,
            "status": "deleted_empty_initial",
            "rows_dropped": 0,
            "nans_filled": 0
        }

    original_row_count = len(df)
    
    # 1. Handle volume == 0 dropping if volume column is present
    has_volume = "volume" in df.columns
    zero_volume_count = 0
    if has_volume:
        # Fix the logic to correctly identify zero volume rows, accounting for NaNs
        zero_volume_mask = (df["volume"] == 0) | (df["volume"].isna())
        zero_volume_count = zero_volume_mask.sum()
        if zero_volume_count > 0:
            df = df[~zero_volume_mask]
            
    # 2. Handle NaNs in OHLC columns
    ohlc_cols = ["open", "high", "low", "close"]
    has_ohlc = all(col in df.columns for col in ohlc_cols)
    nan_rows_filled = 0
    all_nan_count = 0
    
    if has_ohlc and not df.empty:
        # First check if any rows have all OHLC as NaN - we should drop them
        all_nan_mask = df[ohlc_cols].isna().all(axis=1)
        all_nan_count = all_nan_mask.sum()
        if all_nan_count > 0:
            df = df[~all_nan_mask]
            
        if not df.empty:
            # Check for any remaining NaN rows in OHLC
            nan_mask = df[ohlc_cols].isna().any(axis=1)
            nan_indices = df.index[nan_mask]
            nan_rows_filled = len(nan_indices)
            if nan_rows_filled > 0:
                for idx in nan_indices:
                    # Fill low if missing
                    if pd.isna(df.at[idx, "low"]):
                        others = [df.at[idx, col] for col in ["open", "high", "close"] if not pd.isna(df.at[idx, col])]
                        if others:
                            df.at[idx, "low"] = min(others)
                    # Fill high if missing
                    if pd.isna(df.at[idx, "high"]):
                        others = [df.at[idx, col] for col in ["open", "low", "close"] if not pd.isna(df.at[idx, col])]
                        if others:
                            df.at[idx, "high"] = max(others)
                    # Fill open if missing
                    if pd.isna(df.at[idx, "open"]):
                        if not pd.isna(df.at[idx, "low"]):
                            df.at[idx, "open"] = df.at[idx, "low"]
                    # Fill close if missing
                    if pd.isna(df.at[idx, "close"]):
                        if not pd.isna(df.at[idx, "high"]):
                            df.at[idx, "close"] = df.at[idx, "high"]

    remaining_row_count = len(df)
    rows_dropped = original_row_count - remaining_row_count
    
    # Determine if there were any changes
    modified = (rows_dropped > 0) or (nan_rows_filled > 0)
    
    if not modified:
        return {
            "file": file_path,
            "status": "no_change",
            "rows_dropped": 0,
            "nans_filled": 0
        }
        
    if remaining_row_count == 0:
        if not dry_run:
            try:
                os.remove(file_path)
            except Exception as e:
                return {
                    "file": file_path,
                    "status": "error_deleting",
                    "error": str(e),
                    "rows_dropped": 0,
                    "nans_filled": 0
                }
        return {
            "file": file_path,
            "status": "deleted",
            "rows_dropped": original_row_count,
            "nans_filled": int(nan_rows_filled)
        }
    else:
        if not dry_run:
            try:
                df.to_parquet(file_path)
            except Exception as e:
                return {
                    "file": file_path,
                    "status": "error_writing",
                    "error": str(e),
                    "rows_dropped": 0,
                    "nans_filled": 0
                }
        return {
            "file": file_path,
            "status": "updated",
            "rows_dropped": int(rows_dropped),
            "nans_filled": int(nan_rows_filled)
        }

## 3. Helper Function to Run Cleanup (Parallelized)

We use `ThreadPoolExecutor` to speed up the read/write operations because pyarrow releases the GIL, enabling highly parallel file I/O.

In [18]:
def run_cleanup(files, dry_run=True, max_workers=16):
    results = []
    
    print(f"Running cleanup. Mode: {'DRY RUN (No files will be modified/deleted)' if dry_run else 'LIVE RUN (Files will be modified/deleted)'}")
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_parquet_file, f, dry_run=dry_run): f for f in files}
        
        for fut in tqdm(as_completed(futures), total=len(files), desc="Processing Parquets"):
            results.append(fut.result())

    # Generate Summary
    summary = {
        "no_change": 0,
        "updated": 0,
        "deleted": 0,
        "deleted_empty_initial": 0,
        "missing_volume_column": 0,
        "error_reading": 0,
        "error_writing": 0,
        "error_deleting": 0
    }
    
    total_rows_dropped = 0
    total_nans_filled = 0
    errors = []
    updated_files = []
    deleted_files = []
    
    for r in results:
        status = r["status"]
        summary[status] = summary.get(status, 0) + 1
        total_rows_dropped += r.get("rows_dropped", 0)
        total_nans_filled += r.get("nans_filled", 0)
        
        if "error" in r:
            errors.append((r["file"], r["status"], r["error"]))
        elif r["status"] == "updated":
            updated_files.append((r["file"], r.get("rows_dropped", 0), r.get("nans_filled", 0)))
        elif r["status"] in ("deleted", "deleted_empty_initial"):
            deleted_files.append(r["file"])
            
    print("\n--- SUMMARY ---")
    print(f"Total Files Scanned: {len(files):,}")
    print(f"Unchanged Files: {summary['no_change']:,}")
    print(f"Updated Files (Zero volume dropped / NaNs filled): {summary['updated']:,}")
    print(f"Deleted Files (No remaining rows): {summary['deleted']:,}")
    print(f"Deleted Files (Empty initially): {summary['deleted_empty_initial']:,}")
    print(f"Missing Volume Column: {summary.get('missing_volume_column', 0):,}")
    print(f"Read Errors: {summary['error_reading']:,}")
    print(f"Write Errors: {summary['error_writing']:,}")
    print(f"Delete Errors: {summary['error_deleting']:,}")
    print(f"Total Rows Dropped: {total_rows_dropped:,}")
    print(f"Total NaNs Filled: {total_nans_filled:,}")
    
    if errors:
        print(f"\n--- ERRORS ({len(errors)}) ---")
        for f, s, err in errors[:10]:
            print(f"- {f} ({s}): {err}")
        if len(errors) > 10:
            print(f"  ... and {len(errors) - 10} more errors.")
            
    return results, updated_files, deleted_files

## 4. Run Dry Run

We perform a safe dry run first to see how many files contain rows with `volume == 0` and how many would be deleted.

In [19]:
results_dry, updated_dry, deleted_dry = run_cleanup(parquet_files, dry_run=True, max_workers=16)

Running cleanup. Mode: DRY RUN (No files will be modified/deleted)


Processing Parquets:   0%|          | 0/20984 [00:00<?, ?it/s]


--- SUMMARY ---
Total Files Scanned: 20,984
Unchanged Files: 4,808
Updated Files (Zero volume dropped / NaNs filled): 13,901
Deleted Files (No remaining rows): 2,275
Deleted Files (Empty initially): 0
Missing Volume Column: 0
Read Errors: 0
Write Errors: 0
Delete Errors: 0
Total Rows Dropped: 21,944,554
Total NaNs Filled: 0


## 5. View Samples of Affected Files (Optional)

Let's see a few files that would be modified or deleted.

In [20]:
if updated_dry:
    print("Sample of files that will be updated (and number of rows that will be dropped / NaNs filled):")
    for f, count, nans in updated_dry[:5]:
        print(f"- {os.path.basename(f)}: dropped {count} rows, filled {nans} NaNs")
else:
    print("No files require updates.")

if deleted_dry:
    print("\nSample of files that will be deleted entirely:")
    for f in deleted_dry[:5]:
        print(f"- {os.path.basename(f)}")
else:
    print("\nNo files will be deleted entirely.")

Sample of files that will be updated (and number of rows that will be dropped / NaNs filled):
- XWD_TO.parquet: dropped 18 rows, filled 0 NaNs
- XTB_WA.parquet: dropped 49 rows, filled 0 NaNs
- XDPU_L.parquet: dropped 289 rows, filled 0 NaNs
- XAMB_DE.parquet: dropped 76 rows, filled 0 NaNs
- XDWL_DE.parquet: dropped 58 rows, filled 0 NaNs

Sample of files that will be deleted entirely:
- XSPIX.parquet
- XCAPX.parquet
- XPRTX.parquet
- XPMSX.parquet
- AEGFX.parquet


## 6. Run Live Run (Action Required)

Run this cell to actually modify/delete the files.

In [21]:
# Set dry_run=False to execute the cleanup
results_live, updated_live, deleted_live = run_cleanup(parquet_files, dry_run=False, max_workers=16)

Running cleanup. Mode: LIVE RUN (Files will be modified/deleted)


Processing Parquets:   0%|          | 0/20984 [00:00<?, ?it/s]


--- SUMMARY ---
Total Files Scanned: 20,984
Unchanged Files: 4,808
Updated Files (Zero volume dropped / NaNs filled): 13,901
Deleted Files (No remaining rows): 2,275
Deleted Files (Empty initially): 0
Missing Volume Column: 0
Read Errors: 0
Write Errors: 0
Delete Errors: 0
Total Rows Dropped: 21,944,554
Total NaNs Filled: 0
